## Contextual Conpression Retriever

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        '''The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight inot energy.
        Millions of tourists travel to see it every year. The rocks date millions of years.
        '''), metadata={"source": "Doc1"}
    ),
    Document(page_content=(
        '''In medival Europe, castles were buil;t primarily for defense.
        The chrophyll in plant cells captures sunlight during Photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to break castle walls.
        '''), metadata={"source": "Doc2"}
    ),
    Document(page_content=(
        '''Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league.
        '''), metadata={"source": "Doc3"}
    ),
    Document(page_content=(
        '''The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filming involves complex CGI and design.
        '''), metadata={"source": "Doc4"}
    ),
]

In [4]:
# Create as FAISS vector store from the document\
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# Step 2: Create FAISS vector store
vectorstore = FAISS.from_documents(documents=docs, embedding=embedding_model)

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [6]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [10]:
# Set up the compressor using the LLM
llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-20b",
    task="text-generation"
)

llm = ChatHuggingFace(llm=llm)

compressor = LLMChainExtractor.from_llm(llm)

In [11]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [12]:
# Query the retriever
query = "What is photosynthesis"
compressed_result = compression_retriever.invoke(query)

/mnt/d/Academics/Generative AI by CampusX/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1784: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [13]:
for i, doc in enumerate(compressed_result):
    print(f"Document {i+1}:")
    print(f"Title: {doc.metadata.get('title', 'N/A')}")
    # print(f"Content: {doc.page_content[:500]}...")  # Print first 500 characters
    print(f"Content: {doc.page_content}") # Print full article
    print("-" * 80)

Document 1:
Title: N/A
Content: Extracted relevant parts:
Photosynthesis is the process by which green plants convert sunlight inot energy.
--------------------------------------------------------------------------------
Document 2:
Title: N/A
Content: Extracted relevant parts:
The chrophyll in plant cells captures sunlight during Photosynthesis.
--------------------------------------------------------------------------------
Document 3:
Title: N/A
Content: Photosynthesis does not occur in animal cells.
--------------------------------------------------------------------------------


## More Retrievers 
- BM25Retriever
- ParentDocumentRetriever
- SelfQueryRetriever
- TimeWeightVectorRetriever
- MultiVectorRetriever
- EnsembleRetriever
- ArxivRetriever